# Import `deepseek_part2.json` vao Neo4j bang Cypher

Notebook nay doc file JSON, chuan hoa payload, sau do dung `UNWIND` + `MERGE` de import Part -> Chapter -> Crime -> Rule -> Condition -> Penalty vao Neo4j.

In [1]:
import json
import os
from pathlib import Path
from neo4j import GraphDatabase
from dotenv import load_dotenv

load_dotenv()

JSON_PATH = Path(r"C:/Users/Admin/Desktop/DATN/chatbot_rag/deepseek_part2.json")
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print(f"Connected to Neo4j at {NEO4J_URI}")
print(f"JSON path: {JSON_PATH}")

Connected to Neo4j at neo4j+s://5f55db16.databases.neo4j.io
JSON path: C:\Users\Admin\Desktop\DATN\chatbot_rag\deepseek_part2.json


In [15]:
with JSON_PATH.open("r", encoding="utf-8") as f:
    raw_data = json.load(f)

if "parts" in raw_data:
    raw_parts = raw_data["parts"]
elif "part" in raw_data and "chapters" in raw_data:
    raw_parts = [raw_data]
else:
    raise KeyError("JSON phai co key 'parts' hoac dong thoi co 'part' va 'chapters'.")

import_rows = []
for raw_part in raw_parts:
    part_meta = raw_part["part"]
    for raw_chapter in raw_part.get("chapters", []):
        for raw_article in raw_chapter.get("articles", []):
            rules_payload = []
            for raw_rule in raw_article.get("rules", []):
                conditions_payload = []
                for idx, raw_condition in enumerate(raw_rule.get("conditions", []), start=1):
                    value = raw_condition.get("value")
                    conditions_payload.append({
                        "id": f"{raw_rule['rule_id']}_cond_{idx}",
                        "type": raw_condition.get("type"),
                        "field": raw_condition.get("field"),
                        "operator": raw_condition.get("operator"),
                        "min": value.get("min") if isinstance(value, dict) else None,
                        "max": value.get("max") if isinstance(value, dict) else None,
                        "value": value if not isinstance(value, dict) else None,
                        "unit": raw_condition.get("unit"),
                        "text": raw_condition.get("text")
                    })

                raw_penalty = raw_rule.get("penalty", {}) or {}
                penalty_payload = {
                    "id": f"{raw_rule['rule_id']}_penalty",
                    "min": raw_penalty.get("min"),
                    "max": raw_penalty.get("max"),
                    "fine_min": raw_penalty.get("fine_min"),
                    "fine_max": raw_penalty.get("fine_max"),
                    "extra": raw_penalty.get("extra"),
                    "note": raw_penalty.get("note")
                }

                rules_payload.append({
                    "id": raw_rule["rule_id"],
                    "clause": raw_rule.get("clause"),
                    "logic": raw_rule.get("logic"),
                    "priority": raw_rule.get("priority"),
                    "conditions": conditions_payload,
                    "penalty": penalty_payload
                })

            crime = raw_article["crime"]
            import_rows.append({
                "part": {
                    "id": str(part_meta["part_id"]),
                    "name": part_meta.get("name")
                },
                "chapter": {
                    "id": str(raw_chapter["chapter_id"]),
                    "name": raw_chapter.get("name")
                },
                "crime": {
                    "id": str(crime["id"]),
                    "name": crime.get("name"),
                    "article": crime.get("article")
                },
                "rules": rules_payload
            })

print(f"So article rows se import: {len(import_rows)}")
print(import_rows[0]["part"]["name"] if import_rows else "Khong co du lieu")


So article rows se import: 304
Các tội phạm


In [16]:
CONSTRAINT_QUERIES = [
    "CREATE CONSTRAINT part_id IF NOT EXISTS FOR (n:Part) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT chapter_id IF NOT EXISTS FOR (n:Chapter) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT crime_id IF NOT EXISTS FOR (n:Crime) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT rule_id IF NOT EXISTS FOR (n:Rule) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT condition_id IF NOT EXISTS FOR (n:Condition) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT penalty_id IF NOT EXISTS FOR (n:Penalty) REQUIRE n.id IS UNIQUE"
]

IMPORT_QUERY = """
UNWIND $rows AS row
MERGE (p:Part {id: row.part.id})
SET p.name = row.part.name
MERGE (c:Chapter {id: row.chapter.id})
SET c.name = row.chapter.name
MERGE (p)-[:HAS_CHAPTER]->(c)
MERGE (cr:Crime {id: row.crime.id})
SET cr.name = row.crime.name,
    cr.article = row.crime.article
MERGE (c)-[:HAS_ARTICLE]->(cr)
WITH cr, row
UNWIND row.rules AS rule
MERGE (r:Rule {id: rule.id})
SET r.clause = rule.clause,
    r.logic = rule.logic,
    r.priority = rule.priority
MERGE (cr)-[:HAS_RULE]->(r)
WITH r, rule
FOREACH (cond IN rule.conditions |
    MERGE (cnd:Condition {id: cond.id})
    SET cnd.type = cond.type,
        cnd.field = cond.field,
        cnd.operator = cond.operator,
        cnd.min = cond.min,
        cnd.max = cond.max,
        cnd.value = cond.value,
        cnd.unit = cond.unit,
        cnd.text = cond.text
    MERGE (r)-[:HAS_CONDITION]->(cnd)
)
WITH r, rule
MERGE (pn:Penalty {id: rule.penalty.id})
SET pn.min = rule.penalty.min,
    pn.max = rule.penalty.max,
    pn.fine_min = rule.penalty.fine_min,
    pn.fine_max = rule.penalty.fine_max,
    pn.extra = rule.penalty.extra,
    pn.note = rule.penalty.note
MERGE (r)-[:HAS_PENALTY]->(pn)
"""

def run_write(query, params=None):
    with driver.session() as session:
        result = session.run(query, params or {})
        summary = result.consume()
        return summary.counters

def run_write_in_batches(query, rows, batch_size=10):
    totals = {
        "nodes_created": 0,
        "relationships_created": 0,
        "properties_set": 0,
    }
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        counters = run_write(query, {"rows": batch})
        totals["nodes_created"] += counters.nodes_created
        totals["relationships_created"] += counters.relationships_created
        totals["properties_set"] += counters.properties_set
        print(f"Imported batch {start // batch_size + 1}: {len(batch)} articles")
    return totals

for query in CONSTRAINT_QUERIES:
    run_write(query)

print("Constraints ready")


Constraints ready


In [17]:
counters = run_write_in_batches(IMPORT_QUERY, import_rows, batch_size=10)
print(counters)


Imported batch 1: 10 articles
Imported batch 2: 10 articles
Imported batch 3: 10 articles
Imported batch 4: 10 articles
Imported batch 5: 10 articles
Imported batch 6: 10 articles
Imported batch 7: 10 articles
Imported batch 8: 10 articles
Imported batch 9: 10 articles
Imported batch 10: 10 articles
Imported batch 11: 10 articles
Imported batch 12: 10 articles
Imported batch 13: 10 articles
Imported batch 14: 10 articles
Imported batch 15: 10 articles
Imported batch 16: 10 articles
Imported batch 17: 10 articles
Imported batch 18: 10 articles
Imported batch 19: 10 articles
Imported batch 20: 10 articles
Imported batch 21: 10 articles
Imported batch 22: 10 articles
Imported batch 23: 10 articles
Imported batch 24: 10 articles
Imported batch 25: 10 articles
Imported batch 26: 10 articles
Imported batch 27: 10 articles
Imported batch 28: 10 articles
Imported batch 29: 10 articles
Imported batch 30: 10 articles
Imported batch 31: 4 articles
{'nodes_created': 5188, 'relationships_created': 

In [18]:
CHECK_QUERY = """
MATCH (p:Part)
OPTIONAL MATCH (p)-[:HAS_CHAPTER]->(c:Chapter)
OPTIONAL MATCH (c)-[:HAS_ARTICLE]->(cr:Crime)
OPTIONAL MATCH (cr)-[:HAS_RULE]->(r:Rule)
OPTIONAL MATCH (r)-[:HAS_CONDITION]->(cd:Condition)
OPTIONAL MATCH (r)-[:HAS_PENALTY]->(pn:Penalty)
RETURN count(DISTINCT p) AS parts,
       count(DISTINCT c) AS chapters,
       count(DISTINCT cr) AS crimes,
       count(DISTINCT r) AS rules,
       count(DISTINCT cd) AS conditions,
       count(DISTINCT pn) AS penalties
"""

with driver.session() as session:
    summary = session.run(CHECK_QUERY).single()
    print(dict(summary))

{'parts': 1, 'chapters': 13, 'crimes': 304, 'rules': 1081, 'conditions': 2708, 'penalties': 1081}


In [8]:
import json
import os
import re
import unicodedata
from collections import defaultdict
from dotenv import load_dotenv
from openai import OpenAI

GENERIC_TERMS = {
    "toi pham", "toi danh", "hinh phat", "luat hinh su", "bo luat hinh su",
    "trach nhiem hinh su", "truy cuu trach nhiem", "tham nhung"
}

def normalize_text(text):
    text = (text or "").lower().strip()
    text = text.replace("đ", "d").replace("Đ", "D")
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def score_text(term, text, exact_bonus, partial_bonus):
    if not term or not text:
        return 0
    if term == text:
        return exact_bonus
    if term in text or text in term:
        return partial_bonus
    return 0

def safe_number(value, default=0):
    try:
        return float(value)
    except (TypeError, ValueError):
        return default

def parse_money_value(prompt):
    text = normalize_text(prompt)
    compact = text.replace(" ", "")
    m = re.search(r"(\d[\d\.]*)ty", compact)
    if m:
        return float(m.group(1).replace(".", "")) * 1_000_000_000
    m = re.search(r"(\d[\d\.]*)trieu", compact)
    if m:
        return float(m.group(1).replace(".", "")) * 1_000_000
    m = re.search(r"(\d[\d\.]*)nghin", compact)
    if m:
        return float(m.group(1).replace(".", "")) * 1_000
    nums = re.findall(r"\d[\d\.]*", prompt)
    if nums:
        return float(nums[-1].replace(".", ""))
    return None

def parse_money_ranges(text):
    normalized = normalize_text(text)
    matches = re.findall(r"(\d[\d\.]*)", text or "")
    values = [float(x.replace(".", "")) for x in matches]
    multiplier = 1
    if "ty" in normalized:
        multiplier = 1_000_000_000
    elif "trieu" in normalized:
        multiplier = 1_000_000
    elif "nghin" in normalized:
        multiplier = 1_000
    values = [v * multiplier for v in values]
    lower = None
    upper = None
    if len(values) >= 2:
        lower, upper = values[0], values[1]
    elif len(values) == 1:
        if "tu" in normalized and ("duoi" in normalized or "den" in normalized or "tro len" in normalized):
            lower = values[0]
        elif "duoi" in normalized:
            upper = values[0]
        else:
            lower = values[0]
    return lower, upper

def extract_prompt_terms(prompt):
    text = normalize_text(prompt)
    phrases = []
    for token in [
        "tai san nha nuoc", "that thoat", "lang phi", "vu khi quan dung",
        "dong vat", "quy hiem", "giet nguoi", "ma tuy", "hoi lo"
    ]:
        if token in text:
            phrases.append(token)
    return phrases

def infer_amount_bonus(row, amount_value, crime_rules):
    if amount_value is None:
        return 0
    current_clause = safe_number(row.get("clause"), 0)
    rules = crime_rules.get(row["crime_id"], [])
    ranged_rules = []
    for candidate in rules:
        for cond in candidate.get("conditions") or []:
            low, high = parse_money_ranges(cond)
            if low is not None or high is not None:
                ranged_rules.append({
                    "clause": safe_number(candidate.get("clause"), 0),
                    "low": low,
                    "high": high,
                })
    for rr in ranged_rules:
        low = rr["low"]
        high = rr["high"]
        in_range = ((low is None or amount_value >= low) and (high is None or amount_value < high))
        if in_range and current_clause == rr["clause"]:
            return 120
    uppers = [rr["high"] for rr in ranged_rules if rr["high"] is not None]
    if uppers and amount_value >= max(uppers):
        higher_clauses = [safe_number(r.get("clause"), 0) for r in rules if safe_number(r.get("clause"), 0) > max(rr["clause"] for rr in ranged_rules)]
        if higher_clauses:
            target_clause = max(higher_clauses)
            return 110 if current_clause == target_clause else -50
    lowers = [rr["low"] for rr in ranged_rules if rr["low"] is not None]
    if lowers and amount_value < min(lowers):
        lower_clauses = [safe_number(r.get("clause"), 0) for r in rules if safe_number(r.get("clause"), 0) < min(rr["clause"] for rr in ranged_rules)]
        if lower_clauses:
            target_clause = min(lower_clauses)
            return 90 if current_clause == target_clause else -30
    if ranged_rules and current_clause not in [rr["clause"] for rr in ranged_rules]:
        return -20
    return 0

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

if "raw_data" not in globals():
    with JSON_PATH.open("r", encoding="utf-8") as f:
        raw_data = json.load(f)

user_prompt = "Kẻ hiếp dâm người dưới 16 tuổi rủ thêm người cùng hiếp dâm 1 người thì người đó bị tội gì"

extract_prompt = f"""
Ban la tro ly phap ly. Hay tra ve JSON hop le voi 3 truong sau:
- crime_hint: ten toi danh gan nhat theo ngon ngu phap ly
- condition_hint: tinh tiet quan trong nhat de tra cuu, uu tien viet ngan gon theo cach dien dat cua dieu luat
- search_terms: mang 3 den 8 cum tu khoa ngan phuc vu tim kiem
Chi tra ve JSON, khong giai thich.

Cau cua nguoi dung: {user_prompt}
"""

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0,
    messages=[
        {"role": "system", "content": "Chi tra ve JSON hop le."},
        {"role": "user", "content": extract_prompt},
    ],
)

content = resp.choices[0].message.content.strip()
if content.startswith("```json"):
    content = content[7:-3].strip()
elif content.startswith("```"):
    content = content[3:-3].strip()

hints = json.loads(content)
search_terms = hints.get("search_terms", []) or []
search_terms.extend([hints.get("crime_hint", ""), hints.get("condition_hint", "")])
search_terms = [normalize_text(x) for x in search_terms if x and normalize_text(x)]
search_terms = [x for x in dict.fromkeys(search_terms) if x not in GENERIC_TERMS and len(x) >= 4]
crime_hint_norm = normalize_text(hints.get("crime_hint", ""))
condition_hint_norm = normalize_text(hints.get("condition_hint", ""))
amount_value = parse_money_value(user_prompt)
prompt_terms = extract_prompt_terms(user_prompt)

print("Hints tu OpenAI:")
print(json.dumps(hints, ensure_ascii=False, indent=2))
print("Search terms:", search_terms)
print("Prompt terms:", prompt_terms)
print("Amount parsed:", amount_value)

QUERY = """
MATCH (cr:Crime)-[:HAS_RULE]->(r:Rule)-[:HAS_PENALTY]->(p:Penalty)
OPTIONAL MATCH (r)-[:HAS_CONDITION]->(c:Condition)
WITH cr, r, p, collect(DISTINCT c.text) AS conditions
RETURN cr.id AS crime_id,
       cr.name AS crime_name,
       cr.article AS article,
       r.id AS rule_id,
       r.clause AS clause,
       r.logic AS logic,
       r.priority AS priority,
       conditions,
       p.min AS penalty_min,
       p.max AS penalty_max,
       p.extra AS penalty_extra,
       p.note AS penalty_note
"""

with driver.session() as session:
    rows = [dict(record) for record in session.run(QUERY)]

crime_rules = defaultdict(list)
for row in rows:
    crime_rules[row["crime_id"]].append(row)

crime_candidates = []
for crime_id, rule_rows in crime_rules.items():
    crime_name = rule_rows[0]["crime_name"]
    crime_name_norm = normalize_text(crime_name)
    all_conditions = []
    for rr in rule_rows:
        all_conditions.extend([normalize_text(x) for x in (rr.get("conditions") or []) if x])
    crime_score = 0
    for term in prompt_terms:
        crime_score += score_text(term, crime_name_norm, 50, 25)
        crime_score += max([score_text(term, cond, 20, 10) for cond in all_conditions] or [0])
    for term in search_terms:
        crime_score += score_text(term, crime_name_norm, 18, 9)
        crime_score += max([score_text(term, cond, 8, 4) for cond in all_conditions] or [0])
    if crime_hint_norm and score_text(crime_hint_norm, crime_name_norm, 0, 0) > 0:
        crime_score += 5
    if crime_score > 0:
        crime_candidates.append((crime_id, crime_score))

crime_candidates.sort(key=lambda x: -x[1])
best_crime_ids = {cid for cid, _ in crime_candidates[:5]}

scored_rows = []
for row in rows:
    if row["crime_id"] not in best_crime_ids:
        continue
    crime_name_norm = normalize_text(row["crime_name"])
    conditions_norm = [normalize_text(x) for x in (row.get("conditions") or []) if x]
    score = 0
    for term in prompt_terms:
        score += score_text(term, crime_name_norm, 40, 20)
        score += max([score_text(term, cond, 20, 10) for cond in conditions_norm] or [0])
    for term in search_terms:
        score += score_text(term, crime_name_norm, 12, 6)
        score += max([score_text(term, cond, 10, 5) for cond in conditions_norm] or [0])
    score += max([score_text(condition_hint_norm, cond, 60, 30) for cond in conditions_norm] or [0])
    score += infer_amount_bonus(row, amount_value, crime_rules)
    if score > 0:
        row["match_score"] = score
        scored_rows.append(row)

scored_rows.sort(
    key=lambda x: (
        -safe_number(x.get("match_score"), 0),
        safe_number(x.get("priority"), 999),
        -safe_number(x.get("clause"), 0),
        safe_number(x.get("article"), 999999),
    )
)
rows = scored_rows[:5]

print("\nKet qua query Neo4j:")
for row in rows:
    print(json.dumps(row, ensure_ascii=False, indent=2))

def find_rule_details(raw_data, target_rule_id):
    parts = raw_data.get("parts", [raw_data])
    for part in parts:
        for chapter in part.get("chapters", []):
            for article in chapter.get("articles", []):
                crime = article.get("crime", {})
                for rule in article.get("rules", []):
                    if rule.get("rule_id") == target_rule_id:
                        point_conditions = []
                        for cond in rule.get("conditions", []):
                            point_value = cond.get("point") or cond.get("code")
                            if point_value:
                                point_conditions.append({
                                    "point": str(point_value),
                                    "text": cond.get("text", ""),
                                })
                        return {
                            "crime_name": crime.get("name"),
                            "article": crime.get("article"),
                            "clause": rule.get("clause"),
                            "point_conditions": point_conditions,
                            "conditions": [cond.get("text") for cond in rule.get("conditions", []) if cond.get("text")],
                        }
    return None

def choose_best_point(rule_details, user_prompt, condition_hint_norm, search_terms):
    if not rule_details:
        return None
    candidates = rule_details.get("point_conditions", [])
    if not candidates:
        return None
    prompt_norm = normalize_text(user_prompt)
    best = None
    best_score = -1
    for candidate in candidates:
        cond_norm = normalize_text(candidate.get("text", ""))
        score = 0
        score += score_text(condition_hint_norm, cond_norm, 80, 40)
        score += score_text(prompt_norm, cond_norm, 60, 30)
        for term in search_terms:
            score += score_text(term, cond_norm, 20, 10)
        if score > best_score:
            best_score = score
            best = candidate
    return best if best_score > 0 else None

def format_penalty_text(row):
    if row.get("penalty_note"):
        return row["penalty_note"]
    if row.get("penalty_min") is not None and row.get("penalty_max") is not None:
        penalty_text = f"phạt tù từ {row['penalty_min']} đến {row['penalty_max']} năm"
    elif row.get("penalty_min") is not None:
        penalty_text = f"mức phạt từ {row['penalty_min']}"
    else:
        penalty_text = "chịu hình phạt theo quy định"
    if row.get("penalty_extra"):
        penalty_text += f", {row['penalty_extra']}"
    return penalty_text

if rows:
    top = rows[0]
    rule_details = find_rule_details(raw_data, top["rule_id"])
    point_text = ""
    best_point = choose_best_point(rule_details, user_prompt, condition_hint_norm, search_terms)
    if best_point:
        point_text = f", điểm {best_point['point']}"

    explanation = (
        f"Theo trường hợp của bạn, bạn đã vi phạm tại Điều {top['article']}, "
        f"khoản {top['clause']}{point_text}, {top['crime_name']}. "
        f"Bạn sẽ bị {format_penalty_text(top)}."
    )

    print("\nKet luan thu nghiem:")
    print(
        f"Dieu {top['article']} - {top['crime_name']} | khoan {top['clause']} | "
        f"khung hinh phat: {top['penalty_min']} den {top['penalty_max']} nam"
        + (f", {top['penalty_extra']}" if top.get("penalty_extra") else "")
        + (f", note: {top['penalty_note']}" if top.get("penalty_note") else "")
        + f" | match_score={top['match_score']}"
    )
    print("\nGiai thich de doc:")
    print(explanation)
else:
    print("Khong tim thay ket qua phu hop trong Neo4j.")


Hints tu OpenAI:
{
  "crime_hint": "Hiếp dâm người dưới 16 tuổi",
  "condition_hint": "Có đồng phạm trong hành vi hiếp dâm",
  "search_terms": [
    "hiếp dâm",
    "người dưới 16 tuổi",
    "đồng phạm",
    "tội danh",
    "hình phạt",
    "Hiếp dâm người dưới 16 tuổi",
    "Có đồng phạm trong hành vi hiếp dâm"
  ]
}
Search terms: ['hiep dam', 'nguoi duoi 16 tuoi', 'dong pham', 'hiep dam nguoi duoi 16 tuoi', 'co dong pham trong hanh vi hiep dam']
Prompt terms: []
Amount parsed: 1.0

Ket qua query Neo4j:
{
  "crime_id": "142",
  "crime_name": "Tội hiếp dâm người dưới 16 tuổi",
  "article": 142,
  "rule_id": "142_r1",
  "clause": 1,
  "logic": "BASE",
  "priority": 1,
  "conditions": [
    "Dùng vũ lực, đe dọa dùng vũ lực hoặc lợi dụng tình trạng không thể tự vệ được của nạn nhân hoặc thủ đoạn khác giao cấu hoặc thực hiện hành vi quan hệ tình dục khác với người từ đủ 13 tuổi đến dưới 16 tuổi trái với ý muốn của họ",
    "Giao cấu hoặc thực hiện hành vi quan hệ tình dục khác với người dư

In [20]:
# Dong ket noi sau khi import xong
driver.close()